In [5]:
import fastf1
import pandas as pd
import os
import matplotlib
import matplotlib.pyplot as plt
import numpy as np

## Load 2023 Japan Grand Prix for Understanding

In [6]:
# Enable caching to speed up data loading (optional but recommended)
# Cache directory will be created automatically in the scripts directory
if '__file__' in globals():
    # Running as a script
    file_path = os.path.abspath(__file__)
    scripts_dir = os.path.dirname(file_path)
    cache_dir = os.path.join(scripts_dir, 'cache')
else:
    # Running in IPython/interactive mode - use current working directory
    cache_dir = os.path.join(os.getcwd(), 'cache')

# Create cache directory if it doesn't exist
os.makedirs(cache_dir, exist_ok=True)
fastf1.Cache.enable_cache(cache_dir)

# Load a session (example: 2023 Bahrain Grand Prix, Race)
year = 2023
gp = 'Japan'
session_type = 'R'  # R = Race, Q = Qualifying, FP1/FP2/FP3 = Practice

session = fastf1.get_session(year, gp, session_type)
session.load()  # Load all available data for this session

core           INFO 	Loading data for Japanese Grand Prix - Race [v3.7.0]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core        WARNING 	Driver 1 completed the race distance 00:00.076000 before the recorded end of the session.
core           INFO 	Finished loading data for 20 drivers: ['1', '4', '81', '16', '44', '55', '63', '14', '31', '10', '40', '2

## Understanding session.session_status

`session.session_status` is a **DataFrame** containing the session lifecycle - it tracks when the session started, paused, resumed, finished, or had other status changes. This is different from `track_status`, which tracks flags and safety cars.

### What it contains:
- Session lifecycle events with timestamps
- Status changes like "Session started", "Session paused", "Session resumed", "Session finished"
- Overall session state (not track conditions)

### Key Differences from track_status:
- **session_status**: Overall session lifecycle (started, paused, finished)
- **track_status**: Track conditions (flags, safety car, etc.)

### Key Use Cases:
- Determining exact session start and end times
- Identifying if/when a session was paused (e.g., red flag periods)
- Calculating total session duration (excluding pause periods)
- Understanding session interruptions and timeline


In [7]:
# Basic info about the session_status DataFrame
print("=== SESSION STATUS STRUCTURE ===")
print(f"Type: {type(session.session_status)}")
print(f"Shape: {session.session_status.shape}")
print(f"Number of status changes: {len(session.session_status)}")
print(f"\nColumns: {list(session.session_status.columns)}")
print(f"\nIndex: {session.session_status.index.name if session.session_status.index.name else 'Default Index'}")
print(f"\nData types:\n{session.session_status.dtypes}")


=== SESSION STATUS STRUCTURE ===
Type: <class 'pandas.core.frame.DataFrame'>
Shape: (5, 2)
Number of status changes: 5

Columns: ['Time', 'Status']

Index: Default Index

Data types:
Time      timedelta64[ns]
Status             object
dtype: object


In [8]:
# Display the session status data
print("=== SESSION STATUS PREVIEW ===")
session.session_status


=== SESSION STATUS PREVIEW ===


,Time,Status
0,0 days 00:00:07.099000,Inactive
1,0 days 01:03:03.844000,Started
2,0 days 02:34:02.482000,Finished
3,0 days 02:38:01.806000,Finalised
4,0 days 02:38:01.806000,Ends


In [9]:
# Explore the Status column values
print("=== SESSION STATUS VALUES ===")
print(f"Unique status messages: {session.session_status['Status'].unique()}")
print(f"\nStatus message counts:")
print(session.session_status['Status'].value_counts())


=== SESSION STATUS VALUES ===
Unique status messages: ['Inactive' 'Started' 'Finished' 'Finalised' 'Ends']

Status message counts:
Status
Inactive     1
Started      1
Finished     1
Finalised    1
Ends         1
Name: count, dtype: int64


### Column Descriptions:

1. **Time** - Timestamp when the session status changed (relative to session start)
   - Format: Timedelta (e.g., "0 days 00:00:00.000000")
   - Use this to understand the session timeline

2. **Status** - String describing the session status at that moment
   - Common values:
     - **"Session started"** - Session has begun
     - **"Session paused"** - Session was temporarily stopped (e.g., red flag)
     - **"Session resumed"** - Session resumed after being paused
     - **"Session finished"** - Session has ended
     - **"Session aborted"** - Session was cancelled (rare)
   - Status messages may vary slightly depending on the session type

### Important Notes:

- **Session lifecycle**: Typically starts with "Session started" and ends with "Session finished"
- **Pause periods**: If a session is paused (e.g., red flag), you'll see "Session paused" followed by "Session resumed"
- **Time reference**: All times are relative to the session start (Time = 0 at session start)
- **Typically sparse**: Usually only a few rows (typically 2-4: started, possibly paused/resumed, finished)

### Common Status Transitions:

1. **Normal session**: 
   - Started → Finished

2. **Session with interruption**:
   - Started → Paused → Resumed → Finished

3. **Aborted session** (rare):
   - Started → Aborted


In [12]:
# Analyze session status changes
print("=== SESSION STATUS ANALYSIS ===")
print(f"Time range:")
print(f"  First status: {session.session_status['Time'].min()}")
print(f"  Last status: {session.session_status['Time'].max()}")

# Get key status points
df_sorted = session.session_status.sort_values('Time')

print(f"\n=== SESSION TIMELINE ===")
for idx, row in df_sorted.iterrows():
    print(f"{row['Time']}: {row['Status']}")

# Check for specific statuses
has_started = (session.session_status['Status'] == 'Session started').any()
has_finished = (session.session_status['Status'] == 'Session finished').any()
has_paused = (session.session_status['Status'] == 'Session paused').any()
has_resumed = (session.session_status['Status'] == 'Session resumed').any()
has_aborted = (session.session_status['Status'] == 'Session aborted').any()

print(f"\n=== SESSION LIFECYCLE CHECK ===")
print(f"Session started: {has_started}")
print(f"Session finished: {has_finished}")
print(f"Session paused: {has_paused}")
print(f"Session resumed: {has_resumed}")
print(f"Session aborted: {has_aborted}")

# Calculate session duration
if has_started and has_finished:
    start_time = session.session_status[session.session_status['Status'] == 'Session started']['Time'].iloc[0]
    end_time = session.session_status[session.session_status['Status'] == 'Session finished']['Time'].iloc[0]
    total_duration = end_time - start_time
    print(f"\n=== SESSION DURATION ===")
    print(f"Start time: {start_time}")
    print(f"End time: {end_time}")
    print(f"Total duration: {total_duration}")


=== SESSION STATUS ANALYSIS ===
Time range:
  First status: 0 days 00:00:07.099000
  Last status: 0 days 02:38:01.806000

=== SESSION TIMELINE ===
0 days 00:00:07.099000: Inactive
0 days 01:03:03.844000: Started
0 days 02:34:02.482000: Finished
0 days 02:38:01.806000: Finalised
0 days 02:38:01.806000: Ends

=== SESSION LIFECYCLE CHECK ===
Session started: False
Session finished: False
Session paused: False
Session resumed: False
Session aborted: False


## Practical Usage Examples

### 1. Get session start and end times
```python
# Get session start time
start_time = session.session_status[session.session_status['Status'] == 'Session started']['Time'].iloc[0]

# Get session end time
end_time = session.session_status[session.session_status['Status'] == 'Session finished']['Time'].iloc[0]

# Calculate total duration
duration = end_time - start_time
```

### 2. Check if session was interrupted
```python
# Check if session had any pauses
was_paused = (session.session_status['Status'] == 'Session paused').any()
was_resumed = (session.session_status['Status'] == 'Session resumed').any()

if was_paused and was_resumed:
    print("Session had interruptions")
```

### 3. Calculate active session time (excluding pauses)
```python
# Get all pause periods
pauses = session.session_status[session.session_status['Status'] == 'Session paused']
resumes = session.session_status[session.session_status['Status'] == 'Session resumed']

# Calculate total pause duration
# (This requires matching pause/resume pairs)
```

### 4. Filter data based on session status
```python
# Only analyze data when session is active (not paused)
# Filter laps where LapStartTime falls between started and finished
# (and not during pause periods if any)
```

### 5. Using session_status as context
- **Session duration**: Total time session was active
- **Interruptions**: Whether session was paused (indicates red flags or other issues)
- **Timeline validation**: Verify that lap times fall within session active period
- **Data quality**: Helps identify if data issues are related to session interruptions

### 6. Compare with other session attributes
```python
# session.session_status tells you WHEN the session ran
# session.track_status tells you WHAT CONDITIONS were on track
# session.weather_data tells you WEATHER during the session
# All use the same Time reference (relative to session start)
```
